In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd drive/MyDrive/Capstone_Thesis/

/content/drive/MyDrive/Capstone_Thesis


In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 35.1 MB/s eta 0:00:00


In [4]:
!python validate_capstone.py

All model-free capstone validation checks passed.


In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: Tesla T4


In [12]:
!python capstone_robustness.py \
  --task squad \
  --backend hf \
  --model Qwen/Qwen3-4B \
  --n_samples 20 \
  --sample_offset 200 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations any \
  --severities 1 2 3 \
  --typo_difficulty hard \
  --keyword_strategy idf \
  --output results/qwen_squad_pilot.csv

Design: 9 corrupted conditions + 1 matched clean condition(s); 200 total generations.
Loading weights: 100% 398/398 [00:29<00:00, 13.63it/s]
Running clean baseline | mitigation=none on 20 squad examples...
100% 20/20 [00:23<00:00,  1.19s/it]
Running keyboard | any | severity=1 | difficulty=hard | mitigation=none | replicate=1/1
Running keyboard | any | severity=2 | difficulty=hard | mitigation=none | replicate=1/1
Running keyboard | any | severity=3 | difficulty=hard | mitigation=none | replicate=1/1
Running deletion | any | severity=1 | difficulty=hard | mitigation=none | replicate=1/1
Running deletion | any | severity=2 | difficulty=hard | mitigation=none | replicate=1/1
Running deletion | any | severity=3 | difficulty=hard | mitigation=none | replicate=1/1
Running transposition | any | severity=1 | difficulty=hard | mitigation=none | replicate=1/1
Running transposition | any | severity=2 | difficulty=hard | mitigation=none | replicate=1/1
Running transposition | any | severity=3 | d

In [13]:
import pandas as pd

summary = pd.read_csv("results/qwen_squad_pilot_summary.csv")

summary[
    [
        "typo_type",
        "location",
        "severity",
        "error_rate",
        "mean_actual_severity",
        "severity_completion_rate",
        "mean_actual_char_operations",
    ]
].to_string(index=False)

'    typo_type location  severity  error_rate  mean_actual_severity  severity_completion_rate  mean_actual_char_operations\n     deletion      any         1         0.0                  1.00                      1.00                          2.0\n     deletion      any         2         0.0                  2.00                      1.00                          4.0\n     deletion      any         3         0.0                  2.95                      0.95                          5.9\n     keyboard      any         1         0.0                  1.00                      1.00                          2.0\n     keyboard      any         2         0.0                  2.00                      1.00                          4.0\n     keyboard      any         3         0.0                  2.95                      0.95                          5.9\ntransposition      any         1         0.0                  1.00                      1.00                          2.0\ntransposition  

# RUN 1: POSITION-SPECIFIC ROBUSTNESS
# Tests whether typo location changes performance.
# Uses severity 1 and 2 because these levels achieved 100% completion in the pilot.


In [14]:
!python capstone_robustness.py \
  --task squad \
  --backend hf \
  --model Qwen/Qwen3-4B \
  --n_samples 200 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations any \
  --severities 1 2 3 \
  --typo_difficulty hard \
  --keyword_strategy idf \
  --output results/qwen_squad_core.csv

Design: 9 corrupted conditions + 1 matched clean condition(s); 2,000 total generations.
Loading weights: 100% 398/398 [00:29<00:00, 13.44it/s]
Running clean baseline | mitigation=none on 200 squad examples...
100% 200/200 [04:10<00:00,  1.25s/it]
Running keyboard | any | severity=1 | difficulty=hard | mitigation=none | replicate=1/1
Running keyboard | any | severity=2 | difficulty=hard | mitigation=none | replicate=1/1
Running keyboard | any | severity=3 | difficulty=hard | mitigation=none | replicate=1/1
Running deletion | any | severity=1 | difficulty=hard | mitigation=none | replicate=1/1
Running deletion | any | severity=2 | difficulty=hard | mitigation=none | replicate=1/1
Running deletion | any | severity=3 | difficulty=hard | mitigation=none | replicate=1/1
Running transposition | any | severity=1 | difficulty=hard | mitigation=none | replicate=1/1
Running transposition | any | severity=2 | difficulty=hard | mitigation=none | replicate=1/1
Running transposition | any | severity=

In [5]:
from huggingface_hub import login, whoami

login()
print(whoami())

{'type': 'user', 'id': '662ff1640a30d52cb36db0d8', 'name': 'jessicahung', 'fullname': 'Jessica Hung', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': '/avatars/4c22f136af4f7701d9523e48a211ae2c.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-10-23T17:25:56.000Z'}}


In [17]:
!python capstone_robustness.py \
  --task squad \
  --backend hf \
  --model meta-llama/Llama-3.2-3B-Instruct \
  --n_samples 200 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations any \
  --severities 1 2 3 \
  --typo_difficulty hard \
  --keyword_strategy idf \
  --output results/llama_squad_core.csv

Design: 9 corrupted conditions + 1 matched clean condition(s); 2,000 total generations.
config.json: 100% 878/878 [00:00<00:00, 2.25MB/s]
tokenizer_config.json: 100% 54.5k/54.5k [00:00<00:00, 71.2MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:00<00:00, 37.6MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.35MB/s]
model.safetensors.index.json: 100% 20.9k/20.9k [00:00<00:00, 52.5MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.43G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  19% 1.25G/6.43G [01:16<04:24, 19.6MB/s, 19.6MB/s  ]
Reconstructing (incomplete total...):  37% 2.39G/6.43G [01:31<01:31, 44.0MB/s, 44.0MB/s  ]
Reconstructing (incomplete total...):  45% 2.92G/6.43G [01:41<01:48, 32.2MB/s, 44.0MB/s  ]

Reconstructing (incomplete total...):  76% 4.90G/6.43G [01:47<00:16, 92.4MB/s, 74.6MB/s  ]
Reconstructing (incomplete total...):  8

In [18]:
llama_summary = pd.read_csv("results/llama_squad_core_summary.csv")

display(
    llama_summary[
        [
            "typo_type",
            "severity",
            "error_rate",
            "mean_actual_severity",
            "severity_completion_rate",
            "mean_actual_char_operations",
        ]
    ]
)

,typo_type,severity,error_rate,mean_actual_severity,severity_completion_rate,mean_actual_char_operations
0,deletion,1,0.0,1.000,1.000,2.00
1,deletion,2,0.0,2.000,1.000,4.00
2,deletion,3,0.0,2.975,0.975,5.95
3,keyboard,1,0.0,1.000,1.000,2.00
4,keyboard,2,0.0,2.000,1.000,4.00
5,keyboard,3,0.0,2.975,0.975,5.95
6,transposition,1,0.0,1.000,1.000,2.00
7,transposition,2,0.0,2.000,1.000,4.00
8,transposition,3,0.0,2.975,0.975,5.95


In [19]:
!python compare_models.py \
  --inputs \
    results/qwen_squad_core.csv \
    results/llama_squad_core.csv \
  --mitigation none \
  --typo_difficulty hard \
  --bootstrap_samples 5000 \
  --seed 42 \
  --output_dir results/qwen_vs_llama_core

Saved paired cross-model comparison to results/qwen_vs_llama_core. Positive model gaps mean model A degraded more than model B.


In [20]:
gap = pd.read_csv(
    "results/qwen_vs_llama_core/cross_model_gap_by_severity.csv"
)

display(gap)

,model_a,model_b,gap_definition,severity,mean_gap,gap_ci_2.5,gap_ci_97.5,n_examples
0,Qwen3-4B,Llama-3.2-3B-Instruct,drop_model_a_minus_drop_model_b,1,-0.012520,-0.045522,0.021553,200
1,Qwen3-4B,Llama-3.2-3B-Instruct,drop_model_a_minus_drop_model_b,2,-0.072451,-0.116299,-0.029456,200
2,Qwen3-4B,Llama-3.2-3B-Instruct,drop_model_a_minus_drop_model_b,3,-0.074835,-0.122977,-0.027007,200


In [21]:
condition_gaps = pd.read_csv(
    "results/qwen_vs_llama_core/cross_model_gap_by_condition.csv"
)

display(
    condition_gaps[
        [
            "typo_type",
            "severity",
            "mean_gap",
            "gap_ci_2.5",
            "gap_ci_97.5",
            "n_examples",
        ]
    ]
)

,typo_type,severity,mean_gap,gap_ci_2.5,gap_ci_97.5,n_examples
0,deletion,1,-0.020007,-0.058967,0.016849,200
1,deletion,2,-0.063931,-0.117309,-0.011372,200
2,deletion,3,-0.090085,-0.151343,-0.027165,200
3,keyboard,1,-0.038765,-0.086578,0.009136,200
4,keyboard,2,-0.122764,-0.183530,-0.065123,200
5,keyboard,3,-0.056992,-0.115733,0.000935,200
6,transposition,1,0.021212,-0.014727,0.057638,200
7,transposition,2,-0.030657,-0.080563,0.020206,200
8,transposition,3,-0.077429,-0.136356,-0.019541,200


In [22]:
!python capstone_robustness.py \
  --task squad \
  --backend hf \
  --model Qwen/Qwen3-4B \
  --n_samples 200 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations beginning middle end keyword \
  --severities 1 \
  --typo_difficulty standard \
  --keyword_strategy idf \
  --output results/qwen_squad_positions_standard.csv

Design: 12 corrupted conditions + 1 matched clean condition(s); 2,600 total generations.
Loading weights: 100% 398/398 [00:30<00:00, 13.25it/s]
Running clean baseline | mitigation=none on 200 squad examples...
100% 200/200 [04:11<00:00,  1.26s/it]
Running keyboard | beginning | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running keyboard | middle | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running keyboard | end | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running keyboard | keyword | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running deletion | beginning | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running deletion | middle | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running deletion | end | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running deletion | keyword | severity=1 | difficulty=standard | mitigation=none | repl

In [25]:
positions = pd.read_csv(
    "results/qwen_squad_positions_standard_summary.csv"
)

display(
    positions[
        [
            "typo_type",
            "location",
            "severity",
            "error_rate",
            "mean_actual_severity",
            "severity_completion_rate",
            "mean_actual_char_operations",
        ]
    ]
)

print(positions['mean_actual_severity'], positions['mean_actual_char_operations'], positions['error_rate'])

print(
    "Minimum completion rate:",
    positions["severity_completion_rate"].min(),
)

,typo_type,location,severity,error_rate,mean_actual_severity,severity_completion_rate,mean_actual_char_operations
0,deletion,beginning,1,0.0,1.0,1.0,1.0
1,deletion,end,1,0.0,1.0,1.0,1.0
2,deletion,keyword,1,0.0,1.0,1.0,1.0
3,deletion,middle,1,0.0,1.0,1.0,1.0
4,keyboard,beginning,1,0.0,1.0,1.0,1.0
5,keyboard,end,1,0.0,1.0,1.0,1.0
6,keyboard,keyword,1,0.0,1.0,1.0,1.0
7,keyboard,middle,1,0.0,1.0,1.0,1.0
8,transposition,beginning,1,0.0,1.0,1.0,1.0
9,transposition,end,1,0.0,1.0,1.0,1.0


0     1.0
1     1.0
2     1.0
3     1.0
4     1.0
5     1.0
6     1.0
7     1.0
8     1.0
9     1.0
10    1.0
11    1.0
Name: mean_actual_severity, dtype: float64 0     1.0
1     1.0
2     1.0
3     1.0
4     1.0
5     1.0
6     1.0
7     1.0
8     1.0
9     1.0
10    1.0
11    1.0
Name: mean_actual_char_operations, dtype: float64 0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
Name: error_rate, dtype: float64
Minimum completion rate: 1.0


In [26]:
!python capstone_robustness.py \
  --task squad \
  --backend hf \
  --model meta-llama/Llama-3.2-3B-Instruct \
  --n_samples 200 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations beginning middle end keyword \
  --severities 1 \
  --typo_difficulty standard \
  --keyword_strategy idf \
  --output results/llama_squad_positions_standard.csv

Design: 12 corrupted conditions + 1 matched clean condition(s); 2,600 total generations.
Loading weights: 100% 254/254 [00:20<00:00, 12.63it/s]
Running clean baseline | mitigation=none on 200 squad examples...
  0% 0/200 [00:00<?, ?it/s][transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
100% 200/200 [03:02<00:00,  1.09it/s]
Running keyboard | beginning | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running keyboard | middle | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running keyboard | end | severity=1 | difficulty=standard | mitigation=none | replicate=1/1
Running keyboard | ke

In [4]:
!python compare_models.py \
  --inputs \
    results/qwen_squad_positions_standard.csv \
    results/llama_squad_positions_standard.csv \
  --mitigation none \
  --typo_difficulty standard \
  --bootstrap_samples 5000 \
  --seed 42 \
  --output_dir results/qwen_vs_llama_positions_standard

Saved paired cross-model comparison to results/qwen_vs_llama_positions_standard. Positive model gaps mean model A degraded more than model B.


In [6]:
display(
    pd.read_csv(
        "results/qwen_vs_llama_positions_standard/"
        "cross_model_gap_by_severity.csv"
    )
)

,model_a,model_b,gap_definition,severity,mean_gap,gap_ci_2.5,gap_ci_97.5,n_examples
0,Qwen3-4B,Llama-3.2-3B-Instruct,drop_model_a_minus_drop_model_b,1,-0.005053,-0.02405,0.01386,200


In [7]:
position_gaps = pd.read_csv(
    "results/qwen_vs_llama_positions_standard/"
    "cross_model_gap_by_condition.csv"
)

display(
    position_gaps[
        [
            "typo_type",
            "location",
            "mean_gap",
            "gap_ci_2.5",
            "gap_ci_97.5",
            "n_examples",
        ]
    ]
)

,typo_type,location,mean_gap,gap_ci_2.5,gap_ci_97.5,n_examples
0,deletion,beginning,-0.012137,-0.041824,0.017389,200
1,deletion,end,-0.005928,-0.033876,0.021741,200
2,deletion,keyword,0.008757,-0.014991,0.033036,200
3,deletion,middle,-0.015419,-0.045329,0.013633,200
4,keyboard,beginning,0.018954,-0.014134,0.054127,200
5,keyboard,end,-0.021518,-0.058139,0.014061,200
6,keyboard,keyword,0.007360,-0.019010,0.034548,200
7,keyboard,middle,-0.023140,-0.055091,0.008842,200
8,transposition,beginning,0.005412,-0.025464,0.036564,200
9,transposition,end,-0.022043,-0.054552,0.008744,200


In [8]:
!python capstone_visualizations.py \
  --input results/qwen_squad_core.csv \
  --model Qwen/Qwen3-4B \
  --embedding_backend hf \
  --embedding_typo keyboard \
  --embedding_location any \
  --embedding_samples 40 \
  --batch_size 4 \
  --projection pca \
  --mitigation none \
  --typo_difficulty hard \
  --output_dir results/qwen_core_embedding_figures

Saved performance plots to results/qwen_core_embedding_figures
config.json: 100% 726/726 [00:00<00:00, 3.94MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 26.4MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 60.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 123MB/s]

tokenizer.json: downloading bytes:  14% 1.59M/11.4M [00:00<00:03, 2.69MB/s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:00<00:00, 5.47MB/s,  335kB/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:00<00:00, 18.4MB/s, 1.13MB/s  ]
Loading Qwen/Qwen3-4B to extract contextual representations...
model.safetensors.index.json: 100% 32.8k/32.8k [00:00<00:00, 79.9MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0% 0/3 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.99G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/8.04G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   1% 49.8M/8.04G [

In [9]:
!python capstone_visualizations.py \
  --input results/llama_squad_core.csv \
  --model meta-llama/Llama-3.2-3B-Instruct \
  --embedding_backend hf \
  --embedding_typo keyboard \
  --embedding_location any \
  --embedding_samples 40 \
  --batch_size 4 \
  --projection pca \
  --mitigation none \
  --typo_difficulty hard \
  --output_dir results/llama_core_embedding_figures

Saved performance plots to results/llama_core_embedding_figures
Loading meta-llama/Llama-3.2-3B-Instruct to extract contextual representations...
Loading weights: 100% 254/254 [00:08<00:00, 29.90it/s]
Saved embedding and tokenization analysis to results/llama_core_embedding_figures


In [10]:
!python capstone_robustness.py \
  --task squad \
  --backend hf \
  --model meta-llama/Llama-3.2-3B-Instruct \
  --n_samples 200 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations any \
  --severities 2 3 \
  --typo_difficulty hard \
  --keyword_strategy idf \
  --mitigations none self_correct \
  --output results/llama_squad_mitigation.csv

README.md: 100% 7.62k/7.62k [00:00<00:00, 1.95MB/s]

plain_text/train-00000-of-00001.parquet: downloading bytes:  29% 4.19M/14.5M [00:01<00:03, 3.26MB/s]
plain_text/train-00000-of-00001.parquet: downloading bytes: 100% 14.2M/14.2M [00:01<00:00, 9.56MB/s, 1.36MB/s  ]
plain_text/train-00000-of-00001.parquet: reconstructing file: 100% 14.5M/14.5M [00:01<00:00, 9.74MB/s, 1.39MB/s  ]

plain_text/validation-00000-of-00001.par(…): downloading bytes:   0% 0.00/1.82M [00:00<?, ?B/s]
plain_text/validation-00000-of-00001.par(…): downloading bytes: 100% 1.79M/1.79M [00:01<00:00, 1.63MB/s,  174kB/s  ]
plain_text/validation-00000-of-00001.par(…): reconstructing file: 100% 1.82M/1.82M [00:01<00:00, 1.65MB/s,  177kB/s  ]
Generating train split: 100% 87599/87599 [00:00<00:00, 370759.98 examples/s]
Generating validation split: 100% 10570/10570 [00:00<00:00, 463757.16 examples/s]
Design: 12 corrupted conditions + 2 matched clean condition(s); 2,800 total generations.
Loading weights: 100% 254/254 [00:18<

In [12]:
import pandas as pd

mitigation = pd.read_csv(
    "results/llama_squad_mitigation_mitigation_summary.csv"
)

display(
    mitigation[
        [
            "typo_type",
            "severity",
            "n",
            "unmitigated_score",
            "self_correct_score",
            "mean_improvement",
            "improvement_ci_2.5",
            "improvement_ci_97.5",
            "bootstrap_probability_improvement_le_zero",
        ]
    ]
)

,typo_type,severity,n,unmitigated_score,self_correct_score,mean_improvement,improvement_ci_2.5,improvement_ci_97.5,bootstrap_probability_improvement_le_zero
0,deletion,2,200,0.718048,0.717696,-0.000352,-0.035752,0.037579,0.5072
1,deletion,3,200,0.673563,0.653895,-0.019668,-0.061150,0.020979,0.8310
2,keyboard,2,200,0.665385,0.699758,0.034374,0.004831,0.066278,0.0108
3,keyboard,3,200,0.631088,0.637070,0.005982,-0.034273,0.046676,0.3782
4,transposition,2,200,0.779330,0.737863,-0.041467,-0.076770,-0.007987,0.9938
5,transposition,3,200,0.700013,0.664227,-0.035786,-0.080493,0.007543,0.9468


In [13]:
!python capstone_robustness.py \
  --task gsm8k \
  --backend hf \
  --model Qwen/Qwen3-4B \
  --n_samples 100 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations any \
  --severities 1 2 3 \
  --typo_difficulty hard \
  --max_new_tokens 512 \
  --output results/qwen_gsm8k_core.csv

README.md: 100% 7.93k/7.93k [00:00<00:00, 1.78MB/s]

main/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/2.31M [00:00<?, ?B/s]
main/train-00000-of-00001.parquet: downloading bytes: 100% 2.30M/2.30M [00:01<00:00, 2.05MB/s,  223kB/s  ]
main/train-00000-of-00001.parquet: reconstructing file: 100% 2.31M/2.31M [00:01<00:00, 2.05MB/s,  224kB/s  ]

main/test-00000-of-00001.parquet: downloading bytes:   0% 0.00/419k [00:00<?, ?B/s]
main/test-00000-of-00001.parquet: downloading bytes: 100% 419k/419k [00:01<00:00, 399kB/s, 40.8kB/s  ]
main/test-00000-of-00001.parquet: reconstructing file: 100% 419k/419k [00:01<00:00, 399kB/s, 40.8kB/s  ]
Generating train split: 100% 7473/7473 [00:00<00:00, 352144.54 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 308600.82 examples/s]
Design: 9 corrupted conditions + 1 matched clean condition(s); 1,000 total generations.
config.json: 100% 726/726 [00:00<00:00, 2.79MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 8.01MB/s]
v

In [16]:
!python capstone_robustness.py \
  --task gsm8k \
  --backend hf \
  --model meta-llama/Llama-3.2-3B-Instruct \
  --n_samples 100 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations any \
  --severities 1 2 3 \
  --typo_difficulty hard \
  --max_new_tokens 512 \
  --output results/llama_gsm8k_core.csv

Design: 9 corrupted conditions + 1 matched clean condition(s); 1,000 total generations.
Loading weights: 100% 254/254 [00:23<00:00, 10.95it/s]
Running clean baseline | mitigation=none on 100 gsm8k examples...
  0% 0/100 [00:00<?, ?it/s][transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
100% 100/100 [12:32<00:00,  7.53s/it]
Running keyboard | any | severity=1 | difficulty=hard | mitigation=none | replicate=1/1
Running keyboard | any | severity=2 | difficulty=hard | mitigation=none | replicate=1/1
Running keyboard | any | severity=3 | difficulty=hard | mitigation=none | replicate=1/1
Running deletion | any | severity=1 | diffi

In [7]:
display(
    pd.read_csv("results/llama_gsm8k_core_primary_summary.csv")
)

llama_gsm = pd.read_csv("results/llama_gsm8k_core_summary.csv")

display(
    llama_gsm[
        [
            "typo_type",
            "severity",
            "score",
            "error_rate",
            "mean_actual_severity",
            "severity_completion_rate",
            "mean_actual_char_operations",
        ]
    ]
)

,model,task,severity,typo_difficulty,mitigation,n,requested_n,excluded_examples_due_to_errors,clean_score,score,absolute_drop,drop_ci_2.5,drop_ci_97.5,bootstrap_probability_drop_le_zero
0,meta-llama/Llama-3.2-3B-Instruct,gsm8k,1,hard,none,100,100,0,0.79,0.780000,0.010000,-4.333333e-02,0.063333,0.3360
1,meta-llama/Llama-3.2-3B-Instruct,gsm8k,2,hard,none,100,100,0,0.79,0.740000,0.050000,-1.000000e-02,0.110000,0.0448
2,meta-llama/Llama-3.2-3B-Instruct,gsm8k,3,hard,none,100,100,0,0.79,0.726667,0.063333,1.110223e-17,0.126667,0.0222


,typo_type,severity,score,error_rate,mean_actual_severity,severity_completion_rate,mean_actual_char_operations
0,deletion,1,0.77,0.0,1.0,1.0,2.0
1,deletion,2,0.71,0.0,2.0,1.0,4.0
2,deletion,3,0.71,0.0,3.0,1.0,6.0
3,keyboard,1,0.75,0.0,1.0,1.0,2.0
4,keyboard,2,0.73,0.0,2.0,1.0,4.0
5,keyboard,3,0.73,0.0,3.0,1.0,6.0
6,transposition,1,0.82,0.0,1.0,1.0,2.0
7,transposition,2,0.78,0.0,2.0,1.0,4.0
8,transposition,3,0.74,0.0,3.0,1.0,6.0


In [8]:
!python compare_models.py \
  --inputs \
    results/qwen_gsm8k_core.csv \
    results/llama_gsm8k_core.csv \
  --mitigation none \
  --typo_difficulty hard \
  --bootstrap_samples 5000 \
  --seed 42 \
  --output_dir results/qwen_vs_llama_gsm8k

Saved paired cross-model comparison to results/qwen_vs_llama_gsm8k. Positive model gaps mean model A degraded more than model B.


In [10]:
gsm_gap = pd.read_csv(
    "results/qwen_vs_llama_gsm8k/cross_model_gap_by_severity.csv"
)

display(gsm_gap)

,model_a,model_b,gap_definition,severity,mean_gap,gap_ci_2.5,gap_ci_97.5,n_examples
0,Qwen3-4B,Llama-3.2-3B-Instruct,drop_model_a_minus_drop_model_b,1,0.046667,-0.016667,0.113333,100
1,Qwen3-4B,Llama-3.2-3B-Instruct,drop_model_a_minus_drop_model_b,2,0.036667,-0.040000,0.110000,100
2,Qwen3-4B,Llama-3.2-3B-Instruct,drop_model_a_minus_drop_model_b,3,0.060000,-0.003333,0.123333,100


In [9]:
gsm_conditions = pd.read_csv(
    "results/qwen_vs_llama_gsm8k/cross_model_gap_by_condition.csv"
)

display(
    gsm_conditions[
        [
            "typo_type",
            "severity",
            "mean_gap",
            "gap_ci_2.5",
            "gap_ci_97.5",
            "n_examples",
        ]
    ]
)

,typo_type,severity,mean_gap,gap_ci_2.5,gap_ci_97.5,n_examples
0,deletion,1,0.05,-0.03,0.14,100
1,deletion,2,0.02,-0.07,0.11,100
2,deletion,3,0.06,-0.03,0.15,100
3,keyboard,1,0.03,-0.04,0.10,100
4,keyboard,2,0.02,-0.07,0.11,100
5,keyboard,3,0.07,-0.02,0.15,100
6,transposition,1,0.06,0.00,0.13,100
7,transposition,2,0.07,-0.04,0.17,100
8,transposition,3,0.05,-0.04,0.14,100


In [11]:
!python capstone_robustness.py \
  --task squad \
  --backend hf \
  --model Qwen/Qwen3-4B \
  --n_samples 100 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations any \
  --severities 3 \
  --typo_difficulty hard \
  --corruption_replicates 3 \
  --output results/qwen_squad_severity3_replicated.csv

README.md: 100% 7.62k/7.62k [00:00<00:00, 3.12MB/s]

plain_text/train-00000-of-00001.parquet: downloading bytes:  31% 4.49M/14.5M [00:01<00:02, 3.38MB/s]
plain_text/train-00000-of-00001.parquet: downloading bytes: 100% 14.2M/14.2M [00:01<00:00, 10.1MB/s, 1.37MB/s  ]
plain_text/train-00000-of-00001.parquet: reconstructing file: 100% 14.5M/14.5M [00:01<00:00, 10.3MB/s, 1.40MB/s  ]

plain_text/validation-00000-of-00001.par(…): downloading bytes:  98% 1.79M/1.82M [00:01<00:00, 1.60MB/s]
plain_text/validation-00000-of-00001.par(…): downloading bytes: 100% 1.79M/1.79M [00:01<00:00, 1.60MB/s,  174kB/s  ]
plain_text/validation-00000-of-00001.par(…): reconstructing file: 100% 1.82M/1.82M [00:01<00:00, 1.62MB/s,  177kB/s  ]
Generating train split: 100% 87599/87599 [00:00<00:00, 442056.80 examples/s]
Generating validation split: 100% 10570/10570 [00:00<00:00, 338852.70 examples/s]
Design: 9 corrupted conditions + 1 matched clean condition(s); 1,000 total generations.
config.json: 100% 726/726 [00

In [12]:
!python capstone_robustness.py \
  --task squad \
  --backend hf \
  --model meta-llama/Llama-3.2-3B-Instruct \
  --n_samples 100 \
  --seed 42 \
  --typo_types keyboard deletion transposition \
  --match_typo_targets \
  --locations any \
  --severities 3 \
  --typo_difficulty hard \
  --corruption_replicates 3 \
  --output results/llama_squad_severity3_replicated.csv

Design: 9 corrupted conditions + 1 matched clean condition(s); 1,000 total generations.
config.json: 100% 878/878 [00:00<00:00, 3.70MB/s]
tokenizer_config.json: 100% 54.5k/54.5k [00:00<00:00, 86.2MB/s]
tokenizer.json: 100% 9.09M/9.09M [00:01<00:00, 7.24MB/s]
special_tokens_map.json: 100% 296/296 [00:00<00:00, 1.24MB/s]
model.safetensors.index.json: 100% 20.9k/20.9k [00:00<00:00, 59.2MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/1.46G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  42% 2.68G/6.43G [00:20<00:28, 132MB/s,  138MB/s  ]

Reconstructing (incomplete total...):  76% 4.86G/6.43G [00:30<00:06, 228MB/s,  204MB/s  ]
Reconstructing (incomplete total...):  84% 5.39G/6.43G [00:32<00:04, 236MB/s,  228MB/s  ]
Reconstructing (incomplete total...): 100% 6.43G/6.43G [00:43<00:00, 134MB/s,  138MB/s  ]

Fetching 2 files: 100% 2/2 [00:43<00:00, 21

In [13]:
!python compare_models.py \
  --inputs \
    results/qwen_squad_severity3_replicated.csv \
    results/llama_squad_severity3_replicated.csv \
  --mitigation none \
  --typo_difficulty hard \
  --output_dir results/cross_model_squad_severity3_replicated

Saved paired cross-model comparison to results/cross_model_squad_severity3_replicated. Positive model gaps mean model A degraded more than model B.
